## Journey history DB manager demo

This notebook mirrors the example from earlier in the project but now drives the `DBManager` component directly so we can verify inserts, lookups, deletions, and the helper that converts in-memory `JourneyData` objects into persisted records.

In [1]:
import os
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
os.chdir(PROJECT_ROOT)
PROJECT_ROOT

PosixPath('/Users/tombramwell/Documents/Kelder App/kelder_api/kelder_api')

In [2]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

from src.kelder_api.components.db_manager.models import (
    JourneyHistoryRecord,
    JourneyLocation,
)
from src.kelder_api.components.db_manager.service import DBManager
from src.kelder_api.components.log.models import JourneyData

In [3]:
DB_PATH = Path("tests/notebooks/data/journey_history_demo.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

db = DBManager(DB_PATH)
db.clear_history()
db.database_path

INFO | 2025-11-15 10:20:17 | db_manager.service | Journey history database ready at tests/notebooks/data/journey_history_demo.db


PosixPath('tests/notebooks/data/journey_history_demo.db')

In [4]:
base_time = datetime(2025, 1, 10, 10, tzinfo=timezone.utc)
records = [
    JourneyHistoryRecord(
        departure_time=base_time,
        arrival_time=base_time + timedelta(hours=7, minutes=30),
        departure_location=JourneyLocation(latitude="52.13.77", longitude="002.11.27"),
        arrival_location=JourneyLocation(latitude="52.14.30", longitude="002.12.30"),
    ),
    JourneyHistoryRecord(
        departure_time=base_time + timedelta(days=2),
        arrival_time=base_time + timedelta(days=2, hours=6, minutes=10),
        departure_location=JourneyLocation(latitude="51.13.77", longitude="003.11.27"),
        arrival_location=JourneyLocation(latitude="51.14.30", longitude="003.12.30"),
    ),
]
inserted_records = [db.save_trip(record) for record in records]
[record.model_dump(mode="json") for record in inserted_records]

[{'unique_key': 1,
  'departure_time': '2025-01-10T10:00:00Z',
  'arrival_time': '2025-01-10T17:30:00Z',
  'departure_location': {'latitude': '52.13.77', 'longitude': '002.11.27'},
  'arrival_location': {'latitude': '52.14.30', 'longitude': '002.12.30'},
  'duration_seconds': 27000},
 {'unique_key': 2,
  'departure_time': '2025-01-12T10:00:00Z',
  'arrival_time': '2025-01-12T16:10:00Z',
  'departure_location': {'latitude': '51.13.77', 'longitude': '003.11.27'},
  'arrival_location': {'latitude': '51.14.30', 'longitude': '003.12.30'},
  'duration_seconds': 22200}]

In [5]:
[
    (trip.unique_key, trip.departure_time, trip.arrival_time, trip.duration_seconds)
    for trip in db.list_trips()
]

[(2,
  datetime.datetime(2025, 1, 12, 10, 0, tzinfo=datetime.timezone.utc),
  datetime.datetime(2025, 1, 12, 16, 10, tzinfo=datetime.timezone.utc),
  22200),
 (1,
  datetime.datetime(2025, 1, 10, 10, 0, tzinfo=datetime.timezone.utc),
  datetime.datetime(2025, 1, 10, 17, 30, tzinfo=datetime.timezone.utc),
  27000)]

In [7]:
first_id = inserted_records[0].unique_key
db.fetch_trip(first_id).model_dump(mode="json")

{'unique_key': 1,
 'departure_time': '2025-01-10T10:00:00Z',
 'arrival_time': '2025-01-10T17:30:00Z',
 'departure_location': {'latitude': '52.13.77', 'longitude': '002.11.27'},
 'arrival_location': {'latitude': '52.14.30', 'longitude': '002.12.30'},
 'duration_seconds': 27000}

In [8]:
db.latest_trip().model_dump(mode="json")

{'unique_key': 2,
 'departure_time': '2025-01-12T10:00:00Z',
 'arrival_time': '2025-01-12T16:10:00Z',
 'departure_location': {'latitude': '51.13.77', 'longitude': '003.11.27'},
 'arrival_location': {'latitude': '51.14.30', 'longitude': '003.12.30'},
 'duration_seconds': 22200}

In [9]:
db.delete_trip(first_id)
[(trip.unique_key, trip.departure_location.latitude) for trip in db.list_trips()]

[(2, '51.13.77')]

In [10]:
journey_data = JourneyData(
    timestamp=datetime(2025, 2, 5, 9, tzinfo=timezone.utc),
    end_datetime=datetime(2025, 2, 5, 18, 30, tzinfo=timezone.utc),
    start_latitude="50.45.20",
    start_longitude="001.10.30",
    end_latitude="50.55.10",
    end_longitude="001.40.20",
)
db.save_from_journey_data(journey_data)
[
    (trip.unique_key, trip.departure_time.isoformat(), trip.arrival_time.isoformat())
    for trip in db.list_trips()
]

[(3, '2025-02-05T09:00:00+00:00', '2025-02-05T18:30:00+00:00'),
 (2, '2025-01-12T10:00:00+00:00', '2025-01-12T16:10:00+00:00')]

Running all cells leaves the demo database populated with the remaining manually inserted voyage plus the row that originated from `JourneyData`.